# Outline Artifact Viewer (By Run Timestamp)

This notebook prints outline artifacts for a specific run timestamp, grouped by attempts, expansions, revisions, preliminary plan drafts/refinements, and the plan. Each section includes the exact system prompt and the user prompt template used by the LLM for that step, with placeholders where run-specific data is injected.

Prompt blocks can be hidden by setting `SHOW_PROMPTS = False` in the switch cell.


In [1]:
from pathlib import Path
import json
import re

ARTIFACTS_DIR = Path("artifacts")

# Optional: set the run timestamp you want to inspect (from the filename prefix).
# Example: "2026_02_25_02_07_05"
RUN_TIMESTAMP = "2026_02_27_02_54_14"

# Optional: set RUN_ID if multiple runs share the same timestamp.
RUN_ID = None  # e.g., 6

# If RUN_TIMESTAMP is None, pick the most recent outline artifact automatically.
AUTO_PICK_LATEST = True


def _run_prefix_from_path(path: Path) -> str | None:
    match = re.match(
        r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_outline_run_(\d+)_outline_",
        path.name,
    )
    if not match:
        return None
    return f"{match.group(1)}_outline_run_{match.group(2)}"


def _latest_outline_artifact() -> Path | None:
    patterns = [
        "*_outline_run_*_outline_revision*.json",
        "*_outline_run_*_outline_expand*.json",
        "*_outline_run_*_outline_attempt*.json",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(ARTIFACTS_DIR.glob(pattern))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def _pick_run_prefix():
    if RUN_ID is not None and RUN_TIMESTAMP:
        return f"{RUN_TIMESTAMP}_outline_run_{RUN_ID}"

    if not RUN_TIMESTAMP and AUTO_PICK_LATEST:
        latest = _latest_outline_artifact()
        if latest:
            return _run_prefix_from_path(latest)

    if not RUN_TIMESTAMP:
        return None

    candidates = list(ARTIFACTS_DIR.glob(f"{RUN_TIMESTAMP}_outline_run_*_outline_*.json"))
    prefixes = sorted({_run_prefix_from_path(p) for p in candidates} - {None})
    if len(prefixes) == 1:
        return prefixes[0]
    if len(prefixes) > 1:
        latest = max(candidates, key=lambda p: p.stat().st_mtime)
        return _run_prefix_from_path(latest)
    return None


def _print_outline_json(path: Path | None, label: str):
    print(f"=== {label} ===")
    if not path:
        print("(not found)")
        return None
    print(f"File: {path}")
    data = json.loads(path.read_text(encoding="utf-8"))
    outline = data.get("outline")
    if isinstance(outline, list):
        for idx, item in enumerate(outline, 1):
            print(f"{idx}. {item}")
    else:
        print(json.dumps(data, indent=2, ensure_ascii=False))
    return outline if isinstance(outline, list) else None


def _print_outline_group(paths, label):
    print(f"=== {label} ===")
    if not paths:
        print("(not found)")
        return
    for path in paths:
        _print_outline_json(path, path.name)


def _print_markdown(path: Path | None, label: str):
    print(f"=== {label} ===")
    if not path:
        print("(not found)")
        return
    print(f"File: {path}")
    print(path.read_text(encoding="utf-8"))


RUN_PREFIX = _pick_run_prefix()
print(f"RUN_PREFIX: {RUN_PREFIX}")



def _plan_run_prefix_from_path(path: Path) -> str | None:
    match = re.match(
        r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_plan_run_(\d+)_plan_",
        path.name,
    )
    if not match:
        return None
    return f"{match.group(1)}_plan_run_{match.group(2)}"


def _latest_plan_artifact() -> Path | None:
    patterns = [
        "*_plan_run_*_plan_draft_round*.json",
        "*_plan_run_*_plan_refine_round*.json",
        "*_plan_draft_round*.json",
        "*_plan_refine_round*.json",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(ARTIFACTS_DIR.glob(pattern))
    if not candidates:
        return None
    return max(candidates, key=lambda p: p.stat().st_mtime)


def _pick_plan_prefix(outline_prefix: str | None = None):
    if RUN_ID is not None and RUN_TIMESTAMP:
        return f"{RUN_TIMESTAMP}_plan_run_{RUN_ID}"

    if outline_prefix:
        match = re.match(r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})_outline_run_(\d+)", outline_prefix)
        if match:
            return f"{match.group(1)}_plan_run_{match.group(2)}"

    if not RUN_TIMESTAMP and AUTO_PICK_LATEST:
        latest = _latest_plan_artifact()
        if latest:
            return _plan_run_prefix_from_path(latest)

    if not RUN_TIMESTAMP:
        return None

    candidates = list(ARTIFACTS_DIR.glob(f"{RUN_TIMESTAMP}_plan_run_*_plan_*.json"))
    prefixes = sorted({_plan_run_prefix_from_path(p) for p in candidates} - {None})
    if len(prefixes) == 1:
        return prefixes[0]
    if len(prefixes) > 1:
        latest = max(candidates, key=lambda p: p.stat().st_mtime)
        return _plan_run_prefix_from_path(latest)
    return None


def _print_plan_json(path: Path | None, label: str):
    print(f"=== {label} ===")
    if not path:
        print("(not found)")
        return None
    print(f"File: {path}")
    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(data, indent=2, ensure_ascii=False))
    return data


def _print_plan_group(paths, label):
    print(f"=== {label} ===")
    if not paths:
        print("(not found)")
        return
    for path in paths:
        _print_plan_json(path, path.name)


RUN_PREFIX: 2026_02_27_02_54_14_outline_run_1


In [2]:
from IPython.display import Markdown, display

# Switch: set to False to suppress prompt display.
SHOW_PROMPTS = True

def display_prompt(md: str):
    if SHOW_PROMPTS:
        display(Markdown(md))


## Outline Attempts

Input is the topic, methods, selected documents, and KG fact cards assembled during the run. The outliner triggers attempts when it first tries to generate a compliant outline; each attempt is a fresh LLM call saved immediately after parsing. Attempts occur before any length-expansion or revision logic is applied.


In [3]:
display_prompt("""
System prompt used by the LLM:
```text
You are a research planning agent. Produce a research plan (not a report outline). All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include key: outline
- outline MUST be a non-empty array (8-12 major steps).
- Each major step must include 3-5 substeps.
- Each substep should be 2-3 sentences.
- Optional sub-substeps are allowed when helpful.
- Use imperative phrasing (e.g., "Search...", "Compare...", "Investigate...").
- Do not write report section headings.
- Total outline length MUST be 1000-2000 words across titles and substeps in the output language.
- Output must be strict JSON (double quotes, no trailing commas)
- Treat methods as analysis approaches to apply to the topic, not as the topic itself.
- If methods are provided, the research plan MUST explicitly structure steps around those methods.
- All natural-language content MUST be in the requested output language.
- Keep JSON keys in English.
- Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.
- Use ASCII punctuation for JSON (":" and ","), not full-width punctuation.
- When evidence includes confidence or recency, prefer higher-confidence and more recent evidence.
All natural-language content MUST be in the requested output language.

Example (Chinese):
{"outline":[{"title":"查找官方技术报告与发布博客以提取架构与训练细节。","substeps":["列出要检索的官方来源与发布页面。","提取训练规模、数据来源与能力摘要。",{"text":"捕捉部署背景与版本历史。","substeps":["记录发布节奏与更新日志。","整理公开的限制与注意事项。"]}]},{"title":"比较不同模型的上下文长度与长上下文准确性。","substeps":["收集厂商声明与独立基准。","分析长上下文召回与检索性能。",{"text":"识别失效案例。","substeps":["总结常见错误模式。","记录实践中的缓解策略。"]}]}]}

Example (English):
{"outline":[{"title":"Search for official technical reports and release blogs to extract architecture and training details.","substeps":["List official sources and release pages to target.","Extract training scale, data sources, and capability summaries.",{"text":"Capture deployment context and version history.","substeps":["Note release cadence and changelogs.","Record published caveats or limitations."]}]},{"title":"Compare context window sizes and long-context accuracy across models.","substeps":["Collect vendor claims and independent benchmarks.","Analyze long-context recall and retrieval performance.",{"text":"Identify failure cases.","substeps":["Summarize common error patterns.","Note mitigation strategies reported by practitioners."]}]}]}
```

User prompt template sent to the LLM:
```json
{
  "topic": "<topic>",
  "interests": ["<interest>", "..."],
  "methods": ["<method>", "..."],
  "documents": [
    {"title": "<title>", "source_type": "<type>", "source": "<source>", "doc_id": "<id>"}
  ],
  "keywords": ["<keyword>", "..."],
  "output_language": "<language>",
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "language_policy": [
      "All natural-language content MUST be in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.",
      "All natural-language content MUST be in <language>."
    ],
    "outline_expectations": [
      "Include major sections and subtopics to expand later.",
      "Provide 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "List source-backed bullets when available.",
      "Emphasize gaps and future research directions.",
      "Length must be 1000-2000 words in the output language."
    ],
    "method_guidance": [
      "Use methods as analytical lenses applied to the topic and evidence.",
      "Do not treat methods as the topic itself.",
      "If methods are provided, align major steps to the methods and weave method names into the step text.",
      "Do not add bracketed method tags in titles.",
      "If method names are not in <language>, translate them into <language> step titles and include the original in parentheses."
    ],
    "kg_guidance": [
      "If kg_fact_cards is provided, use it to ground claims and outline steps.",
      "Prefer higher-confidence and more recent evidence when shaping the outline.",
      "Prefer evidence-backed relationships over unsupported assertions.",
      "Do not copy evidence snippets verbatim; paraphrase.",
      "When you use kg_fact_cards, add a short citation in-line like 'Sources: title1; title2'.",
      "Do not fabricate sources that are not in kg_fact_cards or documents.",
      "Use trend stats and contradictions in kg_fact_cards to highlight agreements, shifts, and disagreements.",
      "Dates in kg_fact_cards come from document added_at timestamps (ingestion time), not publication dates."
    ]
  },
  "kg_fact_cards": ["<fact card>", "..."]
}
```
""")



System prompt used by the LLM:
```text
You are a research planning agent. Produce a research plan (not a report outline). All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include key: outline
- outline MUST be a non-empty array (8-12 major steps).
- Each major step must include 3-5 substeps.
- Each substep should be 2-3 sentences.
- Optional sub-substeps are allowed when helpful.
- Use imperative phrasing (e.g., "Search...", "Compare...", "Investigate...").
- Do not write report section headings.
- Total outline length MUST be 1000-2000 words across titles and substeps in the output language.
- Output must be strict JSON (double quotes, no trailing commas)
- Treat methods as analysis approaches to apply to the topic, not as the topic itself.
- If methods are provided, the research plan MUST explicitly structure steps around those methods.
- All natural-language content MUST be in the requested output language.
- Keep JSON keys in English.
- Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.
- Use ASCII punctuation for JSON (":" and ","), not full-width punctuation.
- When evidence includes confidence or recency, prefer higher-confidence and more recent evidence.
All natural-language content MUST be in the requested output language.

Example (Chinese):
{"outline":[{"title":"查找官方技术报告与发布博客以提取架构与训练细节。","substeps":["列出要检索的官方来源与发布页面。","提取训练规模、数据来源与能力摘要。",{"text":"捕捉部署背景与版本历史。","substeps":["记录发布节奏与更新日志。","整理公开的限制与注意事项。"]}]},{"title":"比较不同模型的上下文长度与长上下文准确性。","substeps":["收集厂商声明与独立基准。","分析长上下文召回与检索性能。",{"text":"识别失效案例。","substeps":["总结常见错误模式。","记录实践中的缓解策略。"]}]}]}

Example (English):
{"outline":[{"title":"Search for official technical reports and release blogs to extract architecture and training details.","substeps":["List official sources and release pages to target.","Extract training scale, data sources, and capability summaries.",{"text":"Capture deployment context and version history.","substeps":["Note release cadence and changelogs.","Record published caveats or limitations."]}]},{"title":"Compare context window sizes and long-context accuracy across models.","substeps":["Collect vendor claims and independent benchmarks.","Analyze long-context recall and retrieval performance.",{"text":"Identify failure cases.","substeps":["Summarize common error patterns.","Note mitigation strategies reported by practitioners."]}]}]}
```

User prompt template sent to the LLM:
```json
{
  "topic": "<topic>",
  "interests": ["<interest>", "..."],
  "methods": ["<method>", "..."],
  "documents": [
    {"title": "<title>", "source_type": "<type>", "source": "<source>", "doc_id": "<id>"}
  ],
  "keywords": ["<keyword>", "..."],
  "output_language": "<language>",
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "language_policy": [
      "All natural-language content MUST be in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.",
      "All natural-language content MUST be in <language>."
    ],
    "outline_expectations": [
      "Include major sections and subtopics to expand later.",
      "Provide 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "List source-backed bullets when available.",
      "Emphasize gaps and future research directions.",
      "Length must be 1000-2000 words in the output language."
    ],
    "method_guidance": [
      "Use methods as analytical lenses applied to the topic and evidence.",
      "Do not treat methods as the topic itself.",
      "If methods are provided, align major steps to the methods and weave method names into the step text.",
      "Do not add bracketed method tags in titles.",
      "If method names are not in <language>, translate them into <language> step titles and include the original in parentheses."
    ],
    "kg_guidance": [
      "If kg_fact_cards is provided, use it to ground claims and outline steps.",
      "Prefer higher-confidence and more recent evidence when shaping the outline.",
      "Prefer evidence-backed relationships over unsupported assertions.",
      "Do not copy evidence snippets verbatim; paraphrase.",
      "When you use kg_fact_cards, add a short citation in-line like 'Sources: title1; title2'.",
      "Do not fabricate sources that are not in kg_fact_cards or documents.",
      "Use trend stats and contradictions in kg_fact_cards to highlight agreements, shifts, and disagreements.",
      "Dates in kg_fact_cards come from document added_at timestamps (ingestion time), not publication dates."
    ]
  },
  "kg_fact_cards": ["<fact card>", "..."]
}
```


In [4]:
if RUN_PREFIX:
    attempts = sorted(ARTIFACTS_DIR.glob(RUN_PREFIX + "_outline_attempt*.json"))
else:
    attempts = []
_print_outline_group(attempts, "All Outline Attempts")

=== All Outline Attempts ===
=== 2026_02_27_02_54_14_outline_run_1_outline_attempt1.json ===
File: artifacts/2026_02_27_02_54_14_outline_run_1_outline_attempt1.json
1. Systems Engineering: 应用“五看”法进行Agentic Workflow的市场与趋势分析
- 利用“看行业/趋势”维度,深入剖析工作流自动化领域从基于规则的传统引擎向Agentic AI转型的技术演进路径。重点分析“Agentic Business Process Management Systems”中提到的生成式AI与Agentic AI如何引发BPM学科的新一轮进化,以及这一趋势对企业现有IT架构的颠覆性影响(Sources: Agentic Business Process Management Systems; Agentic AI workflows: Trends, examples and best practices)。
- 通过“看市场/客户”维度,调研医疗保健与文档处理等关键垂直领域的自动化需求,识别传统自动化无法满足的复杂决策痛点。详细评估“AI Agents for Intelligent Workflow Automation”中描述的文档处理领域对智能体的需求,以及“Agentic AI-Powered Automation”中指出的医疗流程对降低延迟和提高预测精度的迫切需求(Sources: AI Agents for Intelligent Workflow Automation; Agentic AI-Powered Automation: A New Paradigm for Healthcare Workflow Optimization)。
- 执行“看竞争”分析,对比AI Agents与传统工作流引擎(Conventional Workflow Engines)在底层逻辑上的根本差异。
  - 分析传统工作流引擎依赖的静态规则逻辑(Rule-based logic)在面对非结构化数据时的局限性,对比AI Agents利用机器学习算法处理复杂场景的优势(Sources: AI

## Outline Expansions

Input is the best available outline that is too short. Expansion is triggered when the outline word count falls below the minimum length; the model is asked to add detail without removing content, using a length hint derived from the previous outline. Expansion runs after attempts fail length validation and before revision is enforced.


In [5]:
display_prompt("""
System prompt used by the LLM:
```text
You expand research plan outlines to meet strict minimum length. Return JSON only.
```

User prompt template sent to the LLM:
```json
{
  "previous_outline": ["<outline step>", "..."],
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "requirements": [
      "Length must be at least 1000 words in the output language.",
      "Keep 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "Do not remove content; only expand with additional detail and substeps."
    ],
    "language_guidance": [
      "Write natural-language content in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "structure_guidance": ["<optional guidance>"]
  },
  "length_hint": "<length hint>",
  "language_hint": "<optional language hint>"
}
```
""")



System prompt used by the LLM:
```text
You expand research plan outlines to meet strict minimum length. Return JSON only.
```

User prompt template sent to the LLM:
```json
{
  "previous_outline": ["<outline step>", "..."],
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "requirements": [
      "Length must be at least 1000 words in the output language.",
      "Keep 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "Do not remove content; only expand with additional detail and substeps."
    ],
    "language_guidance": [
      "Write natural-language content in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "structure_guidance": ["<optional guidance>"]
  },
  "length_hint": "<length hint>",
  "language_hint": "<optional language hint>"
}
```


In [6]:
if RUN_PREFIX:
    expands = sorted(ARTIFACTS_DIR.glob(RUN_PREFIX + "_outline_expand*.json"))
else:
    expands = []
_print_outline_group(expands, "All Outline Expansions")

=== All Outline Expansions ===
(not found)


## Outline Revisions

Input is the last outline that still violates length or language constraints. Revision is triggered after attempts and expansions fail to meet the strict word-count target; the model is instructed to rewrite while preserving structure, using a length hint based on the previous outline. Revision occurs near the end of the outline pipeline.


In [7]:
display_prompt("""
System prompt used by the LLM:
```text
You revise research plan outlines to meet strict length constraints. Return JSON only.
```

User prompt template sent to the LLM:
```json
{
  "previous_outline": ["<outline step>", "..."],
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "requirements": [
      "Length must be 1000-2000 words in the output language.",
      "Keep 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "Preserve the original topic coverage and structure; rewrite to fit length."
    ],
    "language_guidance": [
      "Write natural-language content in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "structure_guidance": ["<optional guidance>"]
  },
  "length_hint": "<length hint>",
  "language_hint": "<optional language hint>"
}
```
""")



System prompt used by the LLM:
```text
You revise research plan outlines to meet strict length constraints. Return JSON only.
```

User prompt template sent to the LLM:
```json
{
  "previous_outline": ["<outline step>", "..."],
  "instructions": {
    "output_json_schema": {"outline": ["string"]},
    "requirements": [
      "Length must be 1000-2000 words in the output language.",
      "Keep 8-12 major steps.",
      "Each major step must include 3-5 substeps.",
      "Each substep should be 2-3 sentences.",
      "Preserve the original topic coverage and structure; rewrite to fit length."
    ],
    "language_guidance": [
      "Write natural-language content in <language>.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "structure_guidance": ["<optional guidance>"]
  },
  "length_hint": "<length hint>",
  "language_hint": "<optional language hint>"
}
```


In [8]:
if RUN_PREFIX:
    revisions = sorted(ARTIFACTS_DIR.glob(RUN_PREFIX + "_outline_revision*.json"))
else:
    revisions = []
_print_outline_group(revisions, "All Outline Revisions")

=== All Outline Revisions ===
(not found)


## Plan Draft & Refinement Artifacts

These artifacts capture the parsed plan fields after the draft and refinement steps in each round.
They are saved as JSON in `artifacts/` with filenames like `*_plan_run_*_plan_draft_round*.json` and `*_plan_run_*_plan_refine_round*.json`.


In [9]:
display_prompt("""
System prompt used by the LLM:
```text
You are a research planning agent. Produce a plan that is concise, actionable, and focused on gaps. All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include keys: scope, key_questions, keywords, gaps, notes, readiness
- scope, key_questions, keywords MUST be non-empty arrays (>= 3 items each)
- keywords must include core terms from the topic and interests; include extracted interests if provided
- methods are analysis approaches; use them to frame the plan, not as the topic itself
- gaps and notes may be empty arrays
- retrieval_queries is optional; if provided, it must be a short list of search phrases
- readiness must be one of: "draft", "refined", "ready"
- Output must be strict JSON (double quotes, no trailing commas)
- All natural-language content must be in the requested output language.
```
""")



System prompt used by the LLM:
```text
You are a research planning agent. Produce a plan that is concise, actionable, and focused on gaps. All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include keys: scope, key_questions, keywords, gaps, notes, readiness
- scope, key_questions, keywords MUST be non-empty arrays (>= 3 items each)
- keywords must include core terms from the topic and interests; include extracted interests if provided
- methods are analysis approaches; use them to frame the plan, not as the topic itself
- gaps and notes may be empty arrays
- retrieval_queries is optional; if provided, it must be a short list of search phrases
- readiness must be one of: "draft", "refined", "ready"
- Output must be strict JSON (double quotes, no trailing commas)
- All natural-language content must be in the requested output language.
```


In [10]:
plan_prefix = _pick_plan_prefix(RUN_PREFIX)
print(f"PLAN_PREFIX: {plan_prefix}")

drafts = []
refines = []
if plan_prefix:
    drafts = sorted(ARTIFACTS_DIR.glob(plan_prefix + "_plan_draft_round*.json"))
    refines = sorted(ARTIFACTS_DIR.glob(plan_prefix + "_plan_refine_round*.json"))
else:
    drafts = sorted(ARTIFACTS_DIR.glob("*_plan_draft_round*.json"), key=lambda p: p.stat().st_mtime)
    refines = sorted(ARTIFACTS_DIR.glob("*_plan_refine_round*.json"), key=lambda p: p.stat().st_mtime)
    if AUTO_PICK_LATEST:
        drafts = drafts[-1:]
        refines = refines[-1:]

_print_plan_group(drafts, "Plan Draft Artifacts")
_print_plan_group(refines, "Plan Refinement Artifacts")


PLAN_PREFIX: 2026_02_27_02_54_14_plan_run_1
=== Plan Draft Artifacts ===
(not found)
=== Plan Refinement Artifacts ===
=== 2026_02_27_02_54_14_plan_run_1_plan_refine_round2.json ===
File: artifacts/2026_02_27_02_54_14_plan_run_1_plan_refine_round2.json
{
  "label": "plan_refine_round2",
  "topic": "AI Agent-driven Workflows",
  "created_at": "2026-02-27T07:53:47.523147+00:00",
  "round_number": 2,
  "readiness": "refined",
  "scope": [
    "从传统规则驱动的BPM (Business Process Management) 向结合流程挖掘 (Process Mining) 的 Agentic AI Workflows 演进分析",
    "Agentic Workflow 的核心架构要素: 编排 (Orchestration),多智能体协作 (MAS),人机协同 (HITL) 及流程感知层",
    "垂直行业应用场景深挖: 聚焦医疗保健 (Healthcare) 与文档处理 (Document Processing) 的自动化与决策增强",
    "系统工程视角下的实施: 接口控制 (ICD),数据治理,风险分析 (FMEA) 与从“自动化”到“自主化”的变更管理"
  ],
  "key_questions": [
    "AI Agent 驱动的工作流(概率性)与传统 RPA/工作流引擎(确定性)在底层逻辑与异常处理机制上有何本质区别？",
    "如何将流程挖掘 (Process Mining) 技术集成到 Agentic 系统中,作为 Agent 感知流程状态和发现改进机会的基础层？",
    "在构建 Agentic BPMS 时,应采用何种接口控制 (ICD) 标准以支持多 Agent 间的高效通信与任务

## Plan Artifact

Input is the finalized plan fields and the selected outline. The plan is triggered at the end of the research run and rendered into markdown by `render_plan_md()` after the LLM produces plan fields. Plan generation happens before outline building and refinement rounds if needed.


In [11]:
display_prompt("""
System prompt used by the LLM:
```text
You are a research planning agent. Produce a plan that is concise, actionable, and focused on gaps. All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include keys: scope, key_questions, keywords, gaps, notes, readiness
- scope, key_questions, keywords MUST be non-empty arrays (>= 3 items each)
- keywords must include core terms from the topic and interests; include extracted interests if provided
- methods are analysis approaches; use them to frame the plan, not as the topic itself
- gaps and notes may be empty arrays
- retrieval_queries is optional; if provided, it must be a short list of search phrases
- readiness must be one of: "draft", "refined", "ready"
- Output must be strict JSON (double quotes, no trailing commas)
- All natural-language content MUST be in the requested output language, except retrieval_queries must be in English.
- Keep JSON keys in English.
- Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.
- Use ASCII punctuation for JSON (":" and ","), not full-width punctuation.
- When evidence includes confidence or recency, prefer higher-confidence and more recent evidence.
All natural-language content MUST be in the requested output language.

Example (Chinese):
{"scope":["..."],"key_questions":["..."],"keywords":["..."],"gaps":["..."],"notes":["..."],"readiness":"draft","retrieval_queries":["..."]}

Example (English):
{"scope":["..."],"key_questions":["..."],"keywords":["..."],"gaps":["..."],"notes":["..."],"readiness":"draft","retrieval_queries":["..."]}
```

User prompt template sent to the LLM:
```json
{
  "topic": "<topic>",
  "interests": ["<interest>", "..."],
  "methods": ["<method>", "..."],
  "documents": [
    {"title": "<title>", "source_type": "<type>", "source": "<source>", "doc_id": "<id>"}
  ],
  "output_language": "<language>",
  "instructions": {
    "output_json_schema": {
      "scope": ["string"],
      "key_questions": ["string"],
      "keywords": ["string"],
      "gaps": ["string"],
      "notes": ["string"],
      "retrieval_queries": ["string"],
      "readiness": "draft | refined | ready"
    },
    "language_policy": [
      "All list items must be in <language>.",
      "retrieval_queries must be in English.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "keyword_guidance": [
      "Include key terms from the topic.",
      "Include salient terms from manually added interests.",
      "If extracted_interests are provided, incorporate them.",
      "Do not replace the topic with method names."
    ],
    "retrieval_query_guidance": [
      "If provided, use 2-6 word search phrases.",
      "Avoid negations like 'no' or 'lack of'.",
      "Focus on entities, relationships, and concrete nouns.",
      "Keep to 2-8 total queries.",
      "Queries must be in English."
    ],
    "method_guidance": [
      "Methods are analysis approaches (lenses), not standalone topics.",
      "Apply methods to the topic and evidence."
    ],
    "kg_guidance": [
      "If kg_fact_cards is provided, use it to ground claims and plan steps.",
      "Prefer higher-confidence and more recent evidence when shaping the plan.",
      "Do not copy evidence snippets verbatim; paraphrase.",
      "Do not fabricate sources that are not in kg_fact_cards or documents."
    ]
  },
  "extracted_interests": ["<extracted>", "..."],
  "kg_fact_cards": ["<fact card>", "..."],
  "skill_guidance": ["<skill guidance>"]
}
```
""")



System prompt used by the LLM:
```text
You are a research planning agent. Produce a plan that is concise, actionable, and focused on gaps. All natural-language content MUST be in the requested output language.
Return ONE valid JSON object only. No markdown, no commentary, no extra text.

Requirements:
- Must include keys: scope, key_questions, keywords, gaps, notes, readiness
- scope, key_questions, keywords MUST be non-empty arrays (>= 3 items each)
- keywords must include core terms from the topic and interests; include extracted interests if provided
- methods are analysis approaches; use them to frame the plan, not as the topic itself
- gaps and notes may be empty arrays
- retrieval_queries is optional; if provided, it must be a short list of search phrases
- readiness must be one of: "draft", "refined", "ready"
- Output must be strict JSON (double quotes, no trailing commas)
- All natural-language content MUST be in the requested output language, except retrieval_queries must be in English.
- Keep JSON keys in English.
- Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English.
- Use ASCII punctuation for JSON (":" and ","), not full-width punctuation.
- When evidence includes confidence or recency, prefer higher-confidence and more recent evidence.
All natural-language content MUST be in the requested output language.

Example (Chinese):
{"scope":["..."],"key_questions":["..."],"keywords":["..."],"gaps":["..."],"notes":["..."],"readiness":"draft","retrieval_queries":["..."]}

Example (English):
{"scope":["..."],"key_questions":["..."],"keywords":["..."],"gaps":["..."],"notes":["..."],"readiness":"draft","retrieval_queries":["..."]}
```

User prompt template sent to the LLM:
```json
{
  "topic": "<topic>",
  "interests": ["<interest>", "..."],
  "methods": ["<method>", "..."],
  "documents": [
    {"title": "<title>", "source_type": "<type>", "source": "<source>", "doc_id": "<id>"}
  ],
  "output_language": "<language>",
  "instructions": {
    "output_json_schema": {
      "scope": ["string"],
      "key_questions": ["string"],
      "keywords": ["string"],
      "gaps": ["string"],
      "notes": ["string"],
      "retrieval_queries": ["string"],
      "readiness": "draft | refined | ready"
    },
    "language_policy": [
      "All list items must be in <language>.",
      "retrieval_queries must be in English.",
      "Keep paper titles, dataset names, benchmarks, model names, APIs, and acronyms in English."
    ],
    "keyword_guidance": [
      "Include key terms from the topic.",
      "Include salient terms from manually added interests.",
      "If extracted_interests are provided, incorporate them.",
      "Do not replace the topic with method names."
    ],
    "retrieval_query_guidance": [
      "If provided, use 2-6 word search phrases.",
      "Avoid negations like 'no' or 'lack of'.",
      "Focus on entities, relationships, and concrete nouns.",
      "Keep to 2-8 total queries.",
      "Queries must be in English."
    ],
    "method_guidance": [
      "Methods are analysis approaches (lenses), not standalone topics.",
      "Apply methods to the topic and evidence."
    ],
    "kg_guidance": [
      "If kg_fact_cards is provided, use it to ground claims and plan steps.",
      "Prefer higher-confidence and more recent evidence when shaping the plan.",
      "Do not copy evidence snippets verbatim; paraphrase.",
      "Do not fabricate sources that are not in kg_fact_cards or documents."
    ]
  },
  "extracted_interests": ["<extracted>", "..."],
  "kg_fact_cards": ["<fact card>", "..."],
  "skill_guidance": ["<skill guidance>"]
}
```


In [12]:
plan = None
plan_prefix = None
if RUN_TIMESTAMP:
    plan_prefix = RUN_TIMESTAMP[:16]  # YYYY_MM_DD_HH_MM
elif RUN_PREFIX:
    match = re.match(r"^(\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2})", RUN_PREFIX)
    if match:
        plan_prefix = match.group(1)[:16]

if plan_prefix:
    candidates = list(ARTIFACTS_DIR.glob(plan_prefix + "_plan.md"))
    if candidates:
        plan = max(candidates, key=lambda p: p.stat().st_mtime)
if not plan:
    candidates = list(ARTIFACTS_DIR.glob("*_plan.md"))
    if candidates:
        plan = max(candidates, key=lambda p: p.stat().st_mtime)

_print_markdown(plan, "Plan Artifact")


=== Plan Artifact ===
File: artifacts/2026_02_27_02_56_plan.md
# Research Plan - AI Agent-driven Workflows

- Created at: 2026-02-27T07:53:47.523147+00:00
- Round: 2
- Readiness: refined

## Scope
- 从传统规则驱动的BPM (Business Process Management) 向结合流程挖掘 (Process Mining) 的 Agentic AI Workflows 演进分析
- Agentic Workflow 的核心架构要素: 编排 (Orchestration),多智能体协作 (MAS),人机协同 (HITL) 及流程感知层
- 垂直行业应用场景深挖: 聚焦医疗保健 (Healthcare) 与文档处理 (Document Processing) 的自动化与决策增强
- 系统工程视角下的实施: 接口控制 (ICD),数据治理,风险分析 (FMEA) 与从“自动化”到“自主化”的变更管理

## Key Questions
- AI Agent 驱动的工作流(概率性)与传统 RPA/工作流引擎(确定性)在底层逻辑与异常处理机制上有何本质区别？
- 如何将流程挖掘 (Process Mining) 技术集成到 Agentic 系统中,作为 Agent 感知流程状态和发现改进机会的基础层？
- 在构建 Agentic BPMS 时,应采用何种接口控制 (ICD) 标准以支持多 Agent 间的高效通信与任务切换？
- 针对医疗等高风险行业,如何建立专门的 FMEA (失效模式与影响分析) 框架来评估 Agent '幻觉'或误操作风险？
- 现有的数据基础设施如何进行治理,以满足 Agent 自主决策所需的“清洁数据”标准 (Clean Data)？

## Keywords
- AI Agents, Agentic Workflows, Process Mining, Business Process Management (BPM), Multi-agent Systems (MAS), Orchestration, Human-in-the-loop, He